In [1]:
import os
import json
import pandas as pd
import traceback
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY", "")
if not api_key:
    raise ValueError("Set your Groq API key in the .env file as GROQ_API_KEY=your_key")

os.environ["GROQ_API_KEY"] = api_key

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

In [2]:
import os
from dotenv import load_dotenv

dotenv_path = r'C:\\Users\\22h51\\mcqgen\\.env'
load_dotenv(dotenv_path=dotenv_path)
print('GROQ key loaded:', bool(os.getenv('GROQ_API_KEY')))

GROQ key loaded: True


In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

response = llm.invoke("What is the capital of France?")
print(response.content)

The capital of France is Paris.


In [4]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=api_key
)

prompt = PromptTemplate(
    input_variables=["text"],
    template="Summarize the following:\n\n{text}"
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"text": "Hello World"})
print(result)

The text is a simple greeting that says "Hello World". It's often used as a basic example in programming.


In [5]:
TEMPLATE="""
Text: {text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}
"""

In [6]:
RESPONSE_JSON = {
"1": {
"mcq": "multiple choice question",
"options": {
"a": "choice here",
"b": "choice here",
"c": "choice here",
"d": "choice here",
},
"correct": "correct answer",
},
"2": {
"mcq": "multiple choice question",
"options": {
"a": "choice here",
"b": "choice here",
"c": "choice here",
"d": "choice here",
},
"correct": "correct answer",
},
"3": {
"mcq": "multiple choice question",
"options": {
"a": "choice here",
"b": "choice here",
"c": "choice here",
"d": "choice here",
},
"correct": "correct answer",
},
"4": {
"mcq": "multiple choice question",
"options": {
"a": "choice here",
"b": "choice here",
"c": "choice here",
"d": "choice here",
},
"correct": "correct answer",
},
}

In [12]:
TEMPLATE2="""
Text: {text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}
"""

In [9]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE)


In [14]:
from langchain_core.runnables import RunnableLambda

pipeline = (
    quiz_chain
    | RunnableLambda(
        lambda quiz: {
            "quiz": quiz,
            "text": text,
            "subject": subject
        }
    )
    | review_chain
)

In [15]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

quiz_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
)

review_prompt = PromptTemplate(
    input_variables=["text", "quiz", "subject"],
    template=TEMPLATE2
)

quiz_chain = quiz_prompt | llm | StrOutputParser()
review_chain = review_prompt | llm | StrOutputParser()

In [19]:
with open(file_path, 'r') as file:
 TEXT = file.read()

In [17]:
file_path = r"C:\Users\22h51\mcqgen\data.txt"

In [22]:
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "4": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [29]:
# Run the quiz generation/evaluation using Groq (avoid OpenAI-specific callbacks).
input_data = {
    "text": TEXT,
    "number": NUMBER,
    "subject": SUBJECT,
    "tone": TONE,
    "response_json": json.dumps(RESPONSE_JSON),
}
try:
    # Prefer using the composed chain if available
    response = quiz_chain.invoke(input_data)
except NameError:
    # If chain isn't defined, fall back to a direct LLM prompt invocation
    prompt_text = TEMPLATE.format(text=TEXT, number=NUMBER, subject=SUBJECT, tone=TONE, response_json=json.dumps(RESPONSE_JSON))
    response = llm.invoke(prompt_text)
except Exception as e:
    # Last-resort fallback: try direct LLM invocation and report error
    print('Chain invoke error:', e)
    prompt_text = TEMPLATE.format(text=TEXT, number=NUMBER, subject=SUBJECT, tone=TONE, response_json=json.dumps(RESPONSE_JSON))
    response = llm.invoke(prompt_text)
# Safely print the response
try:
    print(response)
except Exception:
    try:
        print(getattr(response, 'content', response))
    except Exception:
        print('Response received; inspect the `response` variable in the kernel.')

{"1": {"mcq": "Who coined the term machine learning in 1959?", "options": {"a": "Alan Turing", "b": "Arthur Samuel", "c": "Donald Hebb", "d": "Tom M. Mitchell"}, "correct": "b"}, 
"2": {"mcq": "What was the name of the experimental 'learning machine' developed by Raytheon Company in the early 1960s?", "options": {"a": "Cybertron", "b": "AlphaGo", "c": "GANs", "d": "Learning Machines"}, "correct": "a"}, 
"3": {"mcq": "According to Tom M. Mitchell, what is required for a computer program to be said to learn from experience?", "options": {"a": "Its performance at tasks must decrease with experience", "b": "Its performance at tasks must remain the same with experience", "c": "Its performance at tasks must improve with experience", "d": "It must be able to think like a human"}, "correct": "c"}, 
"4": {"mcq": "What type of networks were introduced by Ian Goodfellow and others in 2014?", "options": {"a": "Artificial Neural Networks", "b": "Generative Adversarial Networks (GANs)", "c": "Reinfo

In [26]:
NUMBER=5
SUBJECT="Physics"
TONE="Informative"

In [32]:
# Example: run MCQ generation and return structured JSON with expected keys
sample_text = globals().get('TEXT') or "Machine learning is the study of algorithms that improve automatically through experience."
input_data = {
    "text": sample_text,
    "number": 5,
    "subject": "machine learning",
    "tone": "simple",
    "response_json": json.dumps(RESPONSE_JSON),
}
try:
    if 'quiz_chain' in globals():
        resp = quiz_chain.invoke(input_data)
    else:
        prompt_text = TEMPLATE.format(text=sample_text, number=5, subject='machine learning', tone='simple', response_json=json.dumps(RESPONSE_JSON))
        resp = llm.invoke(prompt_text)
except Exception as e:
    print('Invocation error:', e)
    raise
# Extract text content from response
try:
    content = getattr(resp, 'content', None) or getattr(resp, 'text', None) or str(resp)
except Exception:
    content = str(resp)
# Try to parse JSON from the response
import re
parsed_response = None
try:
    parsed_response = json.loads(content)
except Exception:
    m = re.search(r'(\{\s*"1"[\s\S]*\})', content)
    if m:
        try:
            parsed_response = json.loads(m.group(1))
        except Exception:
            parsed_response = None
# Build structured output
output = {
    "text": sample_text,
    "number": 5,
    "subject": "machine learning",
    "tone": "simple",
    "response": parsed_response if parsed_response is not None else content,
}
print(json.dumps(output, indent=2, ensure_ascii=False))

{
  "text": "The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]\n\nThe earliest machine learning program was introduced in the 1950s, when Samuel invented a computer program that calculated the chance of winning in checkers for each side, but the history of machine learning is rooted in decades of efforts to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] The Hebbian theory of neuron interaction set the groundwork for how many machine learning algorithms work, with connected artificial neurons changing the strength of their connections based on data.[9] Other researchers who have studied human cognitive systems contr

In [34]:
# Build a table-friendly list from the generated quiz JSON
quiz = parsed_response or globals().get('quiz') or {}
quiz_table_data = []
for key, value in quiz.items():
    mcq = value.get('mcq', '')
    options = " | ".join(
        f"{opt}: {opt_val}" for opt, opt_val in value.get('options', {}).items()
    )
    correct = value.get('correct', '')
    quiz_table_data.append({'MCQ': mcq, 'Choices': options, 'Correct': correct})

# Display as a DataFrame if pandas is available, otherwise print the list
try:
    import pandas as pd
    display(pd.DataFrame(quiz_table_data))
except Exception:
    print(quiz_table_data)

,MCQ,Choices,Correct
0,Who coined the term 'machine learning' in 1959?,a: Alan Turing | b: Arthur Samuel | c: Donald ...,b
1,What was the name of the 'learning machine' de...,a: Cybertron | b: AlphaGo | c: Cybernet | d: L...,a
2,"According to Tom M. Mitchell, what is required...",a: Its performance at tasks must decrease with...,c
3,What type of networks were introduced by Ian G...,a: Reinforcement Learning Networks | b: Genera...,b
4,In what year did AlphaGo win against top human...,a: 2014 | b: 2015 | c: 2016 | d: 2017,c
